In [ ]:
!pip install --upgrade torchao -q
!pip install trl --no-deps -q
!pip install -U bitsandbytes>=0.46.1 -q
!pip install -q transformers==4.56.2 bitsandbytes>=0.46.1 peft==0.14.0 -q
!pip install -q -U trl peft -q
!pip install "trl==0.12.2" "peft==0.20.0" -q
!pip install joblib -q

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
import torch
import networkx as nx
import matplotlib.pyplot as plt
import random
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import shutil
import subprocess

REPO_URL = "https://github.com/samuncleML/Bot-Detection.git"
CLONE_DIR = "Bot-Detection"

# Delete the path if it already exists
if os.path.exists(CLONE_DIR):
    print(f"🗑️ Directory '{CLONE_DIR}' exists. Deleting it...")
    shutil.rmtree(CLONE_DIR)

# Clone the repository freshly
print(f"🔄 Cloning {REPO_URL}...")
subprocess.run(["git", "clone", REPO_URL, CLONE_DIR])

In [ ]:
import os
import json
import time
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm

DATA_DIR = "/kaggle/working/Bot-Detection/data"
GRAPH_PATH = f"{DATA_DIR}/Graph.pkl"
LOG_PATH = f"{DATA_DIR}/bot_logs.csv"
INTRA_DONE_PATH = f"{DATA_DIR}/intra_done.json"
BRIDGE_DONE_PATH = f"{DATA_DIR}/bridge_done.json"
BRIDGE_CANDIDATES_PATH = f"{DATA_DIR}/bridge_candidates.pkl"

rng = np.random.default_rng(42)


def split_embedding(s):
    return s.split(",")


def load_bot_df(path):
    bot = pd.read_csv(path)
    bot["embeddings"] = bot["embeddings"].str.replace("[", "").str.replace("]", "")
    bot["embeddings"] = bot["embeddings"].apply(split_embedding)
    bot["embeddings"] = bot["embeddings"].apply(lambda x: np.array(x, dtype=np.float64))
    return bot


def build_embedding_lookup(df, id_col="node_id"):
    lookup = {}
    for node_id, emb in zip(df[id_col], df["embeddings"]):
        v = np.asarray(emb, dtype=np.float32)
        n = np.linalg.norm(v)
        lookup[node_id] = v / n if n > 0 else v
    return lookup


def order_by_similarity(sim):
    n = sim.shape[0]
    start = int(np.argmin(sim.mean(axis=1)))
    order, remaining = [start], set(range(n)) - {start}
    while remaining:
        last = order[-1]
        nxt = max(remaining, key=lambda r: sim[last, r])
        order.append(nxt)
        remaining.remove(nxt)
    return order


def build_ws_intra_community(ids, sim, target_hops=(2, 5), rewire_p=0.12,
                             sim_floor=0.85, k_max=None, rng=None):
    rng = rng if rng is not None else np.random.default_rng(0)
    n = len(ids)
    id_index = {node: k for k, node in enumerate(ids)}
    order = order_by_similarity(sim)
    ordered_ids = [ids[o] for o in order]
    k_max = k_max or max(2, n // 2)

    best = None
    for k_ring in range(1, k_max + 1):
        Gc = nx.Graph()
        Gc.add_nodes_from(ids)
        for a in range(n):
            for step in range(1, k_ring + 1):
                b = (a + step) % n
                u, v = ordered_ids[a], ordered_ids[b]
                if u != v:
                    Gc.add_edge(u, v, cosine_sim=float(sim[id_index[u], id_index[v]]))
        for u, v in list(Gc.edges()):
            if rng.random() < rewire_p:
                cand = [x for x in ids if x != u and not Gc.has_edge(u, x)
                       and sim[id_index[u], id_index[x]] >= sim_floor]
                if cand:
                    w = cand[rng.integers(len(cand))]
                    Gc.remove_edge(u, v)
                    Gc.add_edge(u, w, cosine_sim=float(sim[id_index[u], id_index[w]]))

        avg = nx.average_shortest_path_length(Gc) if n > 1 and nx.is_connected(Gc) else float("inf")
        best = (k_ring, Gc, avg)
        if avg <= target_hops[1]:
            break

    k_ring, Gc, avg = best
    return Gc, k_ring, avg


def _jsonable(x):
    if isinstance(x, tuple):
        return [_jsonable(v) for v in x]
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        return float(x)
    return x


def load_json_set(path):
    if os.path.exists(path):
        with open(path) as f:
            return set(tuple(x) if isinstance(x, list) else x for x in json.load(f))
    return set()


def save_json_set(path, s):
    with open(path, "w") as f:
        json.dump([_jsonable(x) for x in s], f)


def load_graph(path):
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    return nx.Graph()


def save_graph(L, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(L, f)
    os.replace(tmp, path)


def append_log_rows(rows, path):
    if not rows:
        return
    df = pd.DataFrame(rows)
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)


def build_intra_communities(df, L, edge_type="follow", target_hops=(2, 5), rewire_p=0.12,
                            sim_floor=0.85, communities=None, id_col="node_id",
                            comm_col="community", label_col="bot_label", rng=None,
                            done_path=INTRA_DONE_PATH, graph_path=GRAPH_PATH):
    rng = rng if rng is not None else np.random.default_rng(0)
    if communities is None:
        communities = sorted(df[comm_col].unique())

    done = load_json_set(done_path)
    rows = []
    for c in communities:
        if c in done:
            continue
        sub_df = df[df[comm_col] == c]
        ids = sub_df[id_col].tolist()
        emb = np.array(sub_df["embeddings"].tolist(), dtype=np.float32)
        emb /= np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12
        sim = emb @ emb.T

        Gc, k_ring, avg = build_ws_intra_community(ids, sim, target_hops, rewire_p, sim_floor, rng=rng)
        labels = dict(zip(sub_df[id_col], sub_df[label_col])) if label_col in sub_df else {}
        for node in ids:
            L.add_node(node, community=c, bot_label=labels.get(node))
        for u, v, d in Gc.edges(data=True):
            L.add_edge(u, v, weight=d["cosine_sim"], cosine_sim=d["cosine_sim"],
                       interactions=[edge_type], source="intra_ws")

        rows.append({"community": c, "n_nodes": len(ids), "k_ring": k_ring,
                     "avg_hops": avg, "n_edges": Gc.number_of_edges()})
        done.add(c)
        save_json_set(done_path, done)
        save_graph(L, graph_path)
        print(f"intra community {c} done, {len(done)}/{len(communities)}, "
              f"k_ring={k_ring}, avg_hops={avg:.2f}, edges={Gc.number_of_edges()}", flush=True)

    return L, pd.DataFrame(rows)


def min_degree_for_2hop_coverage(n_communities, margin=1.3):
    k = 1
    while k * (k - 1) < n_communities:
        k += 1
    return int(np.ceil(k * margin))


def plan_bridge_topology(communities, avg_degree, rng):
    communities = sorted(communities)
    n = len(communities)
    target_links = int(np.ceil(n * avg_degree / 2))
    links = set()
    attempts = 0
    while len(links) < target_links and attempts < target_links * 20:
        a, b = rng.choice(communities, size=2, replace=False)
        links.add(tuple(sorted((a, b))))
        attempts += 1
    return sorted(links)


# =====================================================================
# Calibrated inter-community bridge builder
# Targets: sampled mean shortest path in [4.77, 4.80] AND >=90% of nodes
# in EVERY community with at least one external edge.
# =====================================================================
TARGET_PATH_LO = 4.77
TARGET_PATH_HI = 4.80
COVERAGE_TARGET = 0.90


# ---------- metrics ----------

def sample_mean_path(G, n_samples=400, seed=0):
    rng = np.random.default_rng(seed)
    nodes = list(G.nodes())
    hops, tries = [], 0
    while len(hops) < n_samples and tries < n_samples * 8:
        u, v = rng.choice(nodes, size=2, replace=False)
        tries += 1
        try:
            hops.append(nx.shortest_path_length(G, u, v))
        except nx.NetworkXNoPath:
            continue
    return float(np.mean(hops)) if hops else float("inf")


def external_degrees(G):
    ext = {}
    for u, v, d in G.edges(data=True):
        if d.get("bridge"):
            ext[u] = ext.get(u, 0) + 1
            ext[v] = ext.get(v, 0) + 1
    return ext


def nodes_by_community(L, communities, comm_col="community"):
    by_comm = {c: [] for c in communities}
    for n, d in L.nodes(data=True):
        c = d.get(comm_col)
        if c in by_comm:
            by_comm[c].append(n)
    return by_comm


def community_coverage(G, by_comm):
    """community -> fraction of its nodes with >=1 external (bridge) edge"""
    ext = external_degrees(G)
    cov = {}
    for c, nodes in by_comm.items():
        if nodes:
            cov[c] = sum(1 for n in nodes if ext.get(n, 0) > 0) / len(nodes)
    return cov


# ---------- bridge plan generation (coverage-aware, no I/O) ----------

def plan_bridge_edges(L, communities, avg_degree, emb_lookup, score_threshold,
                      max_bridge_degree, rng, per_link=3, comm_col="community"):
    """Return a list of accepted bridge edges (u, v, a, b, score).
    Nodes with zero external edges are strongly preferred as endpoints,
    which drives per-community coverage toward 1."""
    links = plan_bridge_topology(communities, avg_degree, rng)
    by_comm = nodes_by_community(L, communities, comm_col)

    bridge_degree = {}
    ext = external_degrees(L)
    for u, v, d in L.edges(data=True):
        if d.get("bridge"):
            bridge_degree[u] = bridge_degree.get(u, 0) + 1
            bridge_degree[v] = bridge_degree.get(v, 0) + 1

    cap = max_bridge_degree
    edges, seen = [], set()

    def pick(nodes):
        avail = [n for n in nodes if bridge_degree.get(n, 0) < cap]
        if not avail:
            return None
        uncovered = [n for n in avail if ext.get(n, 0) == 0]
        pool = uncovered if uncovered else avail
        w = np.array([L.degree(n) + 1 for n in pool], dtype=np.float64)
        w /= w.sum()
        return pool[rng.choice(len(pool), p=w)]

    for a, b in links:
        na, nb = by_comm.get(a, []), by_comm.get(b, [])
        if not na or not nb:
            continue
        for _ in range(per_link):
            u, v = pick(na), pick(nb)
            if u is None or v is None or u == v:
                break
            key = (min(u, v), max(u, v))
            if key in seen or L.has_edge(u, v):
                continue
            eu, ev = emb_lookup.get(u), emb_lookup.get(v)
            score = float(np.dot(eu, ev)) if eu is not None and ev is not None else None
            if score is None or score < score_threshold:
                continue
            seen.add(key)
            edges.append((u, v, a, b, score))
            bridge_degree[u] = bridge_degree.get(u, 0) + 1
            bridge_degree[v] = bridge_degree.get(v, 0) + 1
            ext[u] = ext.get(u, 0) + 1
            ext[v] = ext.get(v, 0) + 1
    return edges


def apply_bridge_edges(G, edges, edge_type):
    for u, v, a, b, score in edges:
        G.add_edge(u, v, weight=score, cosine_sim=score,
                   interactions=[edge_type], source="bridge",
                   bridge=True, community_pair=(a, b))


def evaluate_avg_degree(base_L, communities, avg_degree, emb_lookup,
                        score_threshold, max_bridge_degree, rng,
                        n_samples=400, per_link=3):
    """Build bridges on a COPY of base_L and return (mean_path, min_coverage, edges)."""
    G = base_L.copy()
    edges = plan_bridge_edges(G, communities, avg_degree, emb_lookup,
                              score_threshold, max_bridge_degree, rng, per_link)
    apply_bridge_edges(G, edges, "follow")
    by_comm = nodes_by_community(G, communities)
    cov = community_coverage(G, by_comm)
    m = sample_mean_path(G, n_samples=n_samples, seed=avg_degree)
    return m, (min(cov.values()) if cov else 1.0), edges


# ---------- binary search on avg_degree ----------

def calibrate_avg_degree(base_L, communities, emb_lookup, lo=2, hi=None,
                         score_threshold=0.4, max_bridge_degree=8,
                         max_iter=12, rng=None):
    """Binary search: more community-mesh links -> shorter paths (monotone).
    Returns (chosen_avg_degree, mean_path, min_coverage, edges)."""
    rng = rng if rng is not None else np.random.default_rng(7)
    hi = hi or max(3, len(communities))
    best = None  # closest feasible result seen
    while lo <= hi and max_iter > 0:
        max_iter -= 1
        mid = (lo + hi) // 2
        m, cov, edges = evaluate_avg_degree(base_L, communities, mid, emb_lookup,
                                            score_threshold, max_bridge_degree, rng)
        print(f"[calib] avg_degree={mid}: mean_path={m:.3f}, "
              f"min_community_coverage={cov:.3f}, bridge_edges={len(edges)}", flush=True)
        if best is None or abs(m - TARGET_PATH_LO) < abs(best[1] - TARGET_PATH_LO):
            best = (mid, m, cov, edges)
        if TARGET_PATH_LO <= m <= TARGET_PATH_HI:
            return mid, m, cov, edges
        if m > TARGET_PATH_HI:      # paths too long -> denser mesh
            lo = mid + 1
        else:                        # paths too short -> sparser mesh
            hi = mid - 1
    print("[calib] no integer avg_degree hit the window exactly; "
          f"using closest: avg_degree={best[0]}, mean_path={best[1]:.3f}", flush=True)
    return best


# ---------- coverage repair ----------

def repair_coverage(G, communities, by_comm, emb_lookup, score_threshold,
                    max_bridge_degree, rng, max_passes=4):
    """Greedily add bridges from uncovered nodes until every community
    reaches COVERAGE_TARGET (or no feasible endpoints remain)."""
    for _ in range(max_passes):
        ext = external_degrees(G)
        bd = {}
        for u, v in [(u, v) for u, v, d in G.edges(data=True) if d.get("bridge")]:
            bd[u] = bd.get(u, 0) + 1
            bd[v] = bd.get(v, 0) + 1

        added = 0
        for c in communities:
            nodes = by_comm.get(c, [])
            if not nodes:
                continue
            uncovered = [n for n in nodes if ext.get(n, 0) == 0
                         and bd.get(n, 0) < max_bridge_degree]
            need = int(np.ceil(COVERAGE_TARGET * len(nodes))) - \
                   sum(1 for n in nodes if ext.get(n, 0) > 0)
            if need <= 0:
                continue
            for u in uncovered[:max(need * 2, need)]:   # extra attempts; some fail scoring
                if need <= 0:
                    break
                partners = [n for cc, ns in by_comm.items() if cc != c for n in ns
                            if bd.get(n, 0) < max_bridge_degree]
                partners = [n for n in partners if not G.has_edge(u, n)]
                if not partners:
                    continue
                w = np.array([G.degree(n) + 1 for n in partners], dtype=np.float64)
                w /= w.sum()
                v = partners[rng.choice(len(partners), p=w)]
                eu, ev = emb_lookup.get(u), emb_lookup.get(v)
                score = float(np.dot(eu, ev)) if eu is not None and ev is not None else None
                if score is None or score < score_threshold:
                    continue
                G.add_edge(u, v, weight=score, cosine_sim=score, interactions=["follow"],
                           source="bridge", bridge=True,
                           community_pair=(c, G.nodes[v].get("community")))
                ext[u] = ext.get(u, 0) + 1
                bd[u] = bd.get(u, 0) + 1
                bd[v] = bd.get(v, 0) + 1
                added += 1
                need -= 1
        if added == 0:
            break

    cov = community_coverage(G, by_comm)
    return G, cov


# ---------- main entry point ----------

def build_inter_community_mesh_calibrated(L, df, edge_type="follow",
                                          score_threshold=0.4, max_bridge_degree=8,
                                          communities=None, comm_col="community",
                                          rng=None, graph_path=GRAPH_PATH,
                                          log_path=LOG_PATH):
    rng = rng if rng is not None else np.random.default_rng(1)
    if communities is None:
        communities = sorted(df[comm_col].unique())

    # idempotent: strip any previously added bridges so recalibration is clean
    L.remove_edges_from([(u, v) for u, v, d in L.edges(data=True) if d.get("bridge")])
    base_L = L.copy()

    emb_lookup = build_embedding_lookup(df)

    avg_degree, m, cov, edges = calibrate_avg_degree(
        base_L, communities, emb_lookup,
        score_threshold=score_threshold, max_bridge_degree=max_bridge_degree, rng=rng)

    # materialize the calibrated bridges on the real graph
    apply_bridge_edges(L, edges, edge_type)

    # enforce the >=90% per-community coverage constraint
    by_comm = nodes_by_community(L, communities, comm_col)
    L, cov = repair_coverage(L, communities, by_comm, emb_lookup,
                             score_threshold, max_bridge_degree, rng)

    # final report
    final_mean = sample_mean_path(L, n_samples=1000, seed=0)
    worst = min(cov, key=cov.get)
    print(f"[final] avg_degree={avg_degree}, mean_path={final_mean:.3f} "
          f"(target {TARGET_PATH_LO}-{TARGET_PATH_HI}), total_edges={L.number_of_edges()}", flush=True)
    print(f"[final] worst community {worst}: coverage={cov[worst]:.3f} "
          f"(target >={COVERAGE_TARGET}); communities below target: "
          f"{[c for c, f in cov.items() if f < COVERAGE_TARGET]}", flush=True)

    # log accepted bridges
    rows = [{"community_a": a, "community_b": b, "src": u, "target": v,
             "path_score": s, "accepted": True} for u, v, a, b, s in edges]
    append_log_rows(rows, log_path)

    save_graph(L, graph_path)
    return L, {"avg_degree": avg_degree, "mean_path": final_mean, "coverage": cov}


def validate_hops(L, n_samples=500, seed=0, hops_range=(3, 6)):
    rng = np.random.default_rng(seed)
    nodes = list(L.nodes())
    hops, tries = [], 0
    while len(hops) < n_samples and tries < n_samples * 8:
        u, v = rng.choice(nodes, size=2, replace=False)
        tries += 1
        try:
            hops.append(nx.shortest_path_length(L, u, v))
        except nx.NetworkXNoPath:
            continue
    if not hops:
        return pd.Series(dtype=int)
    hops = np.array(hops)
    dist = pd.Series(hops).value_counts().sort_index()
    print(hops.mean(), np.median(hops), ((hops >= hops_range[0]) & (hops <= hops_range[1])).mean(), flush=True)
    print(dist, flush=True)
    return dist


def plot_full_mesh(L, seed=42, node_size=8, max_nodes=5000, save_path=None):
    if L.number_of_nodes() > max_nodes:
        rng_s = np.random.default_rng(seed)
        keep = set(rng_s.choice(list(L.nodes()), size=max_nodes, replace=False))
        G = L.subgraph(keep)
    else:
        G = L

    communities_list = sorted({d["community"] for _, d in G.nodes(data=True) if d.get("community") is not None})
    palette = cm.get_cmap("hsv", max(len(communities_list), 1))
    color_of = {c: palette(i / max(len(communities_list), 1)) for i, c in enumerate(communities_list)}
    node_colors = [color_of.get(d.get("community"), (0.6, 0.6, 0.6, 1)) for _, d in G.nodes(data=True)]

    normal_edges = [(u, v) for u, v, d in G.edges(data=True) if not d.get("bridge")]
    bridge_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get("bridge")]

    pos = nx.spring_layout(G, seed=seed, k=1.2 / np.sqrt(G.number_of_nodes()), iterations=50)

    fig, ax = plt.subplots(figsize=(13, 13))
    nx.draw_networkx_edges(G, pos, edgelist=normal_edges, ax=ax, alpha=0.15, edge_color="gray", width=0.4)
    nx.draw_networkx_edges(G, pos, edgelist=bridge_edges, ax=ax, alpha=0.6, edge_color="black", width=0.6)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_size, node_color=node_colors, linewidths=0)
    ax.set_title(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
                f"{len(communities_list)} communities, {len(bridge_edges)} bridges")
    ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()


def plot_community_mesh(L, save_path=None, seed=42):
    C = nx.Graph()
    for u, v, d in L.edges(data=True):
        if not d.get("bridge"):
            continue
        cu, cv = L.nodes[u].get("community"), L.nodes[v].get("community")
        if cu is None or cv is None or cu == cv:
            continue
        C.add_edge(cu, cv)

    if C.number_of_edges() == 0:
        print("no bridge edges found", flush=True)
        return C

    pos = nx.spring_layout(C, seed=seed, k=1.5 / np.sqrt(C.number_of_nodes()))
    deg = dict(C.degree())

    fig, ax = plt.subplots(figsize=(9, 9))
    nx.draw_networkx_edges(C, pos, ax=ax, alpha=0.4, edge_color="#2a6fdb", width=1.0)
    nx.draw_networkx_nodes(C, pos, ax=ax, node_size=[80 + 40 * deg[n] for n in C.nodes()],
                           node_color=list(deg.values()), cmap="viridis")
    nx.draw_networkx_labels(C, pos, ax=ax, font_size=8)
    try:
        diam = nx.diameter(C)
    except nx.NetworkXError:
        diam = "inf"
    ax.set_title(f"community mesh: {C.number_of_nodes()} communities, {C.number_of_edges()} bridges, diameter {diam}")
    ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()
    return C


def bridge_degree_report(L):
    deg = {}
    for u, v, d in L.edges(data=True):
        if d.get("bridge"):
            deg[u] = deg.get(u, 0) + 1
            deg[v] = deg.get(v, 0) + 1
    if not deg:
        print("no bridge edges yet", flush=True)
        return
    vals = np.array(list(deg.values()))
    print(f"nodes with a bridge: {len(deg)}/{L.number_of_nodes()}, "
          f"max={vals.max()}, mean={vals.mean():.2f}, p95={np.percentile(vals,95):.1f}", flush=True)


# =====================================================================
# Main
# =====================================================================
if __name__ == "__main__":
    os.makedirs(DATA_DIR, exist_ok=True)
    bot = load_bot_df(f"{DATA_DIR}/bot_netdf.csv")
    communities = sorted(bot["community"].unique())

    L = load_graph(GRAPH_PATH)

    # dense intra-community small-world nets (2-5 hops)
    L, intra_summary = build_intra_communities(bot, L, target_hops=(2, 5), communities=communities, rng=rng)
    print(intra_summary, flush=True)

    # calibrated bridges: mean path in [4.77, 4.80] AND >=90% external-edge
    # coverage in every community
    L, bridge_stats = build_inter_community_mesh_calibrated(
        L, bot, score_threshold=0.4, max_bridge_degree=8, communities=communities, rng=rng)
    print(bridge_stats["avg_degree"], bridge_stats["mean_path"], flush=True)

    validate_hops(L, n_samples=1000)
    bridge_degree_report(L)

    save_graph(L, GRAPH_PATH)

    plot_full_mesh(L, save_path=f"{DATA_DIR}/graph_mesh.png")
    plot_community_mesh(L, save_path=f"{DATA_DIR}/community_mesh.png")